In [83]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [84]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

all_science_data = pd.read_csv('science_data_large.csv')
print(all_science_data[0:15])

all_science_data.info() # Print the number of rows and columns


    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## Part 2. Splitting the dataset

In [85]:
# Take the pandas dataset and split it into our features (X) and label (y)
feature_series = all_science_data[['Temperature °C', 'Mols KCL']]
label_series = all_science_data['Size nm^3']

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
x_train, x_test, y_train, y_test = train_test_split(feature_series, label_series, test_size=0.1)

## Part 3. Perform a Linear Regression

In [86]:
# Use sklearn to train a model on the training set

regression = LinearRegression().fit(x_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model

sample_data_point = pd.DataFrame([[400,400]], columns=['Temperature °C', 'Mols KCL'])
print(f'It predicts the size to be: {regression.predict(sample_data_point)[0]}')

# Report on the score for that model, in your own words (markdown, not code) explain what the score means

print(f'The score on the test data is: {regression.score(x_test, y_test)}')

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
print(f'Intercept: {regression.intercept_}, Coefficients: {regression.coef_}')

It predicts the size to be: 346753.8753065939
The score on the test data is: 0.8840492990726521
Intercept: -415347.24714574387, Coefficients: [ 871.87023523 1033.3825709 ]


The score is a measure of the reduction of error by regression compared to just using an average of each of the labels. So in this case, the score is essentially telling us that the regression reduced the error by 85.120% as compared to just averaging each of the data points to predict the next one. In this way, its a measure of how effective a particular model fits to the data.  

From the intercept and coefficients, we get the equation $h(x) = -418790 + 874.04x_1 + 1040.8x_2$, where $x_1$ is the temperature and $x_2$ is the Mols KCL.

## Part 4. Use Cross Validation

In [87]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
scores = cross_val_score(regression, feature_series, label_series)

# Report on their finding and their significance
print(f'The scores were: {scores}, with mean of {scores.mean()}')

The scores were: [0.83918826 0.87051239 0.85871066 0.87202623 0.84364641], with mean of 0.8568167899144437


These scores tell us that even by taking different samples of our data to use for our tests, the minimum percent gain in prediction accuracy was around 83.919%, and had an average of 85.682%. This means that our model seems to be a fairly good fit in other cases besides just the specific test cases we previously used.

## Part 5. Using Polynomial Regression

In [88]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
pf = PolynomialFeatures(degree=2)
augmented_features = pf.fit_transform(feature_series)

aug_train_x, aug_test_x, aug_train_y, aug_test_y = train_test_split(augmented_features, label_series, test_size=0.1)
aug_reg = LinearRegression().fit(aug_train_x, aug_train_y)

# Report on the metrics and output the resultant equation as you did in Part 3.
print(f'The score for the augmented data set is: {aug_reg.score(aug_test_x, aug_test_y)}')
print(f'The intercept is: {aug_reg.intercept_}, and the coefficients are: {aug_reg.coef_}')
print(f'The feature names for augmented data are: {pf.get_feature_names_out()}')


The score for the augmented data set is: 1.0
The intercept is: 1.8050253856927156e-05, and the coefficients are: [ 0.00000000e+00  1.20000000e+01 -1.18448289e-07  5.26161233e-12
  2.00000000e+00  2.85714287e-02]
The feature names for augmented data are: ['1' 'Temperature °C' 'Mols KCL' 'Temperature °C^2'
 'Temperature °C Mols KCL' 'Mols KCL^2']


We can see that the score for this augmented data set is 100%, which implies we completely removed the error compared to just averaging the values. Since the error is completely removed, this regression must fit the test data points perfectly.

For the hypothesis function, we get $h(x)=0.00002 + 0x_1 + 12x_2 + 0x_3 + 0x_4 + 2x_5 + 0.02857x_6$, which simplifies to $0.00002 + 12x_2 + 2x_5 + 0.02857x_6$. By the feature names for the augmented data, we can see $x_2$ is the temperature, $x_5$ is the temperature times the Mols KCL, and $x_6$ is the Mols KCL squared.